In [1]:
list.of.packages <- c("ggplot2", "Rcpp", "grf", "caret", "mltools", "rpart", "minpack.lm", "doParallel", "rattle", "anytime","rlist")
list.of.packages <- c(list.of.packages, "zoo", "dtw", "foreach", "evaluate","rlist","data.table","plyr","dplyr", "fs")


new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='/home/zwang937/local/R_libs', repos="http://cran.us.r-project.org", dependencies = TRUE, INSTALL_opts = '--no-lock')


lapply(list.of.packages, require, character.only = TRUE)

# Register the cluster
registerDoParallel(cores=detectCores())

Loading required package: ggplot2

Loading required package: Rcpp

Loading required package: grf

Loading required package: caret

Loading required package: lattice

Loading required package: mltools

Loading required package: rpart

Loading required package: minpack.lm

Loading required package: doParallel

Loading required package: foreach

Loading required package: iterators

Loading required package: parallel

Loading required package: rattle

Loading required package: tibble

Loading required package: bitops

Rattle: A free graphical interface for data science with R.
Version 5.5.1 Copyright (c) 2006-2021 Togaware Pty Ltd.
Type 'rattle()' to shake, rattle, and roll your data.

Loading required package: anytime

Loading required package: rlist

Loading required package: zoo


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Loading required package: dtw

Loading required package: proxy


Attaching package: ‘proxy’


Th

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

[[20]]
[1] TRUE

In [2]:
print("Setting up directory path")
Individual_RDS_directory <- "../../../data/output/individual_county_grf_windowsize=2_numtrees=100"

# Get a list of all subfolder names (integer names)
subfolder_names <- dir_ls(Individual_RDS_directory, type = "dir", recurse = FALSE, full_path = FALSE)

# Create an empty list to store the loaded objects
#loaded_objects <- list()
print("Loading Objects")
loaded_objects <- foreach(subfolder_name = subfolder_names, .errorhandling="pass") %dopar% {
  # Get the full path to the subfolder
  directory_name <- basename(strsplit(subfolder_name, "/")[[1]])
  fips_string <- directory_name[length(directory_name)]
  # Get the file names of the RDS objects in the subfolder
  #subfolder_feature_importance <- list()
  subfolder_feature_importance <- foreach(cutoff =seq(100, 1000, by = 100), .errorhandling="pass") %do% {
      fname <- paste0("grf_individual_county_fips=",fips_string,"_cutoff=",toString(cutoff),".rds")
      #print(fname)
      rds_file_path <- file.path(subfolder_name, fname)
      # Try reading the RDS file and extract feature importance
      tryCatch({
        loaded_object <- readRDS(rds_file_path)
        feature_importance <- t(grf::variable_importance(loaded_object))
        feature_names <- colnames(loaded_object$X.orig)
        colnames(feature_importance) <- feature_names
        feature_importance
      }, error = function(e) {
        # Return NULL in case of error
        NULL
      })
  }
  #print(subfolder_feature_importance)
  # Return the loaded objects for this subfolder
  #loaded_objects[[subfolder_name]] <- subfolder_feature_importance
  compact(subfolder_feature_importance)
}

[1] "Setting up directory path"
[1] "Loading Objects"


In [3]:
loaded_objects <- loaded_objects[lengths(loaded_objects) > 0]


In [4]:
unlisted_loaded_objects<-unlist(loaded_objects, recursive = FALSE)

In [5]:
length(unlisted_loaded_objects)

[1] 28501

In [6]:
unlisted_loaded_objects[lengths(unlisted_loaded_objects) != 467]

list()

In [7]:
colnames(unlisted_loaded_objects[[1]])

[1] "fips"                                                                                                                            
  [2] "log_rolled_cases.y"                                                                                                              
  [3] "LAT"                                                                                                                             
  [4] "LON"                                                                                                                             
  [5] "AREA_SQMI"                                                                                                                       
  [6] "E_TOTPOP"                                                                                                                        
  [7] "E_HU"                                                                                                                            
  [8] "E_HH"                                                                                                                            
  [9] "E_POV"                                                                                                                           
 [10] "E_UNEMP"                                                                                                                         
 [11] "E_PCI"                                                                                                                           
 [12] "E_NOHSDP"                                                                                                                        
 [13] "E_AGE65"                                                                                                                         
 [14] "E_AGE17"                                                                                                                         
 [15] "E_DISABL"                                                                                                                        
 [16] "E_SNGPNT"                                                                                                                        
 [17] "E_MINRTY"                                                                                                                        
 [18] "E_LIMENG"                                                                                                                        
 [19] "E_MUNIT"                                                                                                                         
 [20] "E_MOBILE"                                                                                                                        
 [21] "E_CROWD"                                                                                                                         
 [22] "E_NOVEH"                                                                                                                         
 [23] "E_GROUPQ"                                                                                                                        
 [24] "EP_POV"                                                                                                                          
 [25] "MP_POV"                                                                                                                          
 [26] "EP_UNEMP"                                                                                                                        
 [27] "MP_UNEMP"                                                                                                                        
 [28] "EP_PCI"                                                                                                                          
 [29] "MP_PCI"                                                                                                                          
 [30] "EP_NOHSDP"            

In [8]:
result_matrix <- matrix(NA, nrow = length(unlisted_loaded_objects), ncol = lengths(unlisted_loaded_objects)[1])
for (i in 1:length(unlisted_loaded_objects)) {
  result_matrix[i, ] <- as.vector(unlisted_loaded_objects[[i]])
}

In [9]:
colnames(result_matrix) <- colnames(unlisted_loaded_objects[[1]])

In [10]:
my_datatable <- data.table(result_matrix)
my_datatable

fips,log_rolled_cases.y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,⋯,Variant % CH.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5,cutoff,t0.lm,r.lm
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0.000000000,0,0,0,0,0,0,0,0,⋯,0.0000000000,0.0000000000,0,0,0.0000000000,0.0000000000,0.0000000000,0.0000000000,0.00000000,0.00000000
0,0.000000000,0,0,0,0,0,0,0,0,⋯,0.0070243902,0.0000000000,0,0,0.0000000000,0.0031358885,0.0000000000,0.0000000000,0.01404878,0.08667596
0,0.004797284,0,0,0,0,0,0,0,0,⋯,0.0056995346,0.0022336374,0,0,0.0000000000,0.0000000000,0.0009518144,0.0070243902,0.11170026,0.11421434
0,0.007592321,0,0,0,0,0,0,0,0,⋯,0.0006614303,0.0012725345,0,0,0.0000000000,0.0000000000,0.0010209870,0.0019842910,0.09091406,0.07539066
0,0.009783991,0,0,0,0,0,0,0,0,⋯,0.0000000000,0.0008130081,0,0,0.0000000000,0.0000000000,0.0000000000,0.0153893130,0.05769878,0.07191826
0,0.002594059,0,0,0,0,0,0,0,0,⋯,0.0000000000,0.0000000000,0,0,0.0000000000,0.0019153735,0.0000000000,0.0172008424,0.09710663,0.07918552
0,0.010139472,0,0,0,0,0,0,0,0,⋯,0.0000000000,0.0006846384,0,0,0.0000000000,0.0000000000,0.0000000000,0.0012195122,0.08884719,0.06212363
0,0.008126326,0,0,0,0,0,0,0,0,⋯,0.0000000000,0.0000000000,0,0,0.0000000000,0.0000000000,0.0009977827,0.0013341672,0.06501180,0.07816695
0,0.007145713,0,0,0,0,0,0,0,0,⋯,0.0000000000,0.0010452962,0,0,0.0000000000,0.0012794882,0.0000000000,0.0070243902,0.07436385,0.08193530


In [11]:
fwrite(my_datatable, "individual_grf_feature_importance.csv", row.names=FALSE)